<a href="https://colab.research.google.com/github/thuynguyenhuit/hocsau/blob/main/ResNet%2BLSTM_15k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import json
import zipfile
import random
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

# =========================================================
# 1. ĐƯỜNG DẪN
# =========================================================
DRIVE_DIR = "/content/drive/MyDrive/VQA"

QUESTION_ZIP = os.path.join(
    DRIVE_DIR,
    "v2_Questions_Train_mscoco.zip"
)

ANNOTATION_ZIP = os.path.join(
    DRIVE_DIR,
    "v2_Annotations_Train_mscoco.zip"
)

# =========================================================
# 2. LOAD QUESTIONS
# =========================================================
print("Loading questions...")

with zipfile.ZipFile(QUESTION_ZIP, 'r') as z:
    with z.open(
        'v2_OpenEnded_mscoco_train2014_questions.json'
    ) as f:

        questions_data = json.load(f)['questions']

# =========================================================
# 3. LOAD ANSWERS
# =========================================================
print("Loading annotations...")

with zipfile.ZipFile(ANNOTATION_ZIP, 'r') as z:
    with z.open(
        'v2_mscoco_train2014_annotations.json'
    ) as f:

        annotations_data = json.load(f)['annotations']

# =========================================================
# 4. GHÉP QUESTION + ANSWER
# =========================================================
print("Merging QA pairs...")

qa_pairs = {}

for q in questions_data:

    qa_pairs[q['question_id']] = {
        "image_id": q["image_id"],
        "question": q["question"]
    }

for ann in annotations_data:

    qid = ann["question_id"]

    if qid in qa_pairs:

        qa_pairs[qid]["answer"] = ann[
            "multiple_choice_answer"
        ].lower()

# =========================================================
# 5. CHUYỂN THÀNH LIST
# =========================================================
dataset_samples = list(qa_pairs.values())

print("Total original samples:", len(dataset_samples))

# =========================================================
# 6. LẤY TOP 1000 ANSWERS
# =========================================================
print("Selecting top 1000 answers...")

all_answers = [
    sample["answer"]
    for sample in dataset_samples
]

answer_counter = Counter(all_answers)

TOP_K = 1000

top_answers = set([
    ans for ans, _ in answer_counter.most_common(TOP_K)
])

print("Top answers:", len(top_answers))

# =========================================================
# 7. GIỮ CHỈ CÁC SAMPLE THUỘC TOP ANSWERS
# =========================================================
filtered_samples = [
    sample for sample in dataset_samples
    if sample["answer"] in top_answers
]

print("Filtered samples:", len(filtered_samples))

# =========================================================
# 8. RANDOM 15,000 MẪU
# =========================================================
NUM_SAMPLES = 15000

random.seed(42)

filtered_samples = random.sample(
    filtered_samples,
    NUM_SAMPLES
)

print("Random selected:", len(filtered_samples))

# =========================================================
# 9. TẠO DATAFRAME
# =========================================================
rows = []

for idx, sample in enumerate(filtered_samples):

    rows.append({
        "stt": idx + 1,
        "image_id": sample["image_id"],
        "question": sample["question"],
        "answer": sample["answer"]
    })

df = pd.DataFrame(rows)

print(df.head())

# =========================================================
# 10. CHIA TRAIN / VAL / TEST
# =========================================================
# 70% train
# 15% val
# 15% test

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

print("\nTRAIN:", len(train_df))
print("VAL:", len(val_df))
print("TEST:", len(test_df))
# =========================================================
# ĐẾM SỐ LƯỢNG ẢNH UNIQUE
# =========================================================

train_images = train_df["image_id"].nunique()
val_images = val_df["image_id"].nunique()
test_images = test_df["image_id"].nunique()

print("\n📸 SỐ LƯỢNG ẢNH UNIQUE")
print(f"Train images: {train_images}")
print(f"Val images:   {val_images}")
print(f"Test images:  {test_images}")
# =========================================================
# 11. LƯU CSV
# =========================================================
train_path = os.path.join(DRIVE_DIR, "train_15k.csv")
val_path = os.path.join(DRIVE_DIR, "val_15k.csv")
test_path = os.path.join(DRIVE_DIR, "test_15k.csv")

train_df.to_csv(
    train_path,
    index=False,
    encoding="utf-8-sig"
)

val_df.to_csv(
    val_path,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    test_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ Saved files:")
print(train_path)
print(val_path)
print(test_path)

# =========================================================
# 12. LƯU TOP ANSWERS
# =========================================================
top_answers_path = os.path.join(
    DRIVE_DIR,
    "top_answers_1000.txt"
)

with open(top_answers_path, "w", encoding="utf-8") as f:

    for ans in sorted(top_answers):
        f.write(ans + "\n")

print("\n✅ Saved top answers:")
print(top_answers_path)

Loading questions...
Loading annotations...
Merging QA pairs...
Total original samples: 443757
Selecting top 1000 answers...
Top answers: 1000
Filtered samples: 388158
Random selected: 15000
   stt  image_id                                question        answer
0    1    243262               Is this animal contained?            no
1    2    149993                      Is it a sunny day?           yes
2    3    105063                What is the man pulling?           wii
3    4    570801          What is the laptop leaning on?  toilet paper
4    5    209772  How many cars only have one headlight?             1

TRAIN: 10500
VAL: 2250
TEST: 2250

📸 SỐ LƯỢNG ẢNH UNIQUE
Train images: 9501
Val images:   2201
Test images:  2203

✅ Saved files:
/content/drive/MyDrive/VQA/train_15k.csv
/content/drive/MyDrive/VQA/val_15k.csv
/content/drive/MyDrive/VQA/test_15k.csv

✅ Saved top answers:
/content/drive/MyDrive/VQA/top_answers_1000.txt


In [2]:
#load lại dữ liệu để tiếp tục các bước sau
import pandas as pd
train_df = pd.read_csv("/content/drive/MyDrive/VQA/Data/train_15k.csv")
val_df = pd.read_csv("/content/drive/MyDrive/VQA/Data/val_15k.csv")
test_df = pd.read_csv("/content/drive/MyDrive/VQA/Data/test_15k.csv")

In [10]:
#BUILD VOCABULARY
import re
from collections import Counter

# =========================================================
# CLEAN TEXT
# =========================================================
def clean_text(text):

    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)

    return text.split()

# =========================================================
# BUILD QUESTION VOCAB
# =========================================================
word_counter = Counter()

for q in train_df["question"]:

    tokens = clean_text(q)
    word_counter.update(tokens)

# special tokens
word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in word_counter:

    word2idx[word] = len(word2idx)

idx2word = {
    idx: word
    for word, idx in word2idx.items()
}

print("VOCAB SIZE:", len(word2idx))

# =========================================================
# BUILD ANSWER VOCAB
# =========================================================
answers = sorted(
    train_df["answer"].unique()
)

# thêm UNK token
ans2idx = {
    "<UNK>": 0
}

# bắt đầu từ 1
for idx, ans in enumerate(answers):

    ans2idx[ans] = idx + 1

# reverse mapping
idx2ans = {
    idx: ans
    for ans, idx in ans2idx.items()
}

print("NUM ANSWERS:", len(ans2idx))

VOCAB SIZE: 3402
NUM ANSWERS: 799


In [11]:
import zipfile
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

# =========================================================
# CONFIG
# =========================================================
MAX_LEN = 20

IMG_ZIP_PATH = "/content/drive/MyDrive/VQA/train2014.zip"

# =========================================================
# DATASET
# =========================================================
class VQADataset(Dataset):

    def __init__(
        self,
        dataframe,
        word2idx,
        ans2idx,
        zip_path,
        max_len=20
    ):

        self.df = dataframe.reset_index(drop=True)

        self.word2idx = word2idx
        self.ans2idx = ans2idx

        self.zip_path = zip_path
        self.max_len = max_len

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):

        return len(self.df)

    def tokenize_question(self, question):

        tokens = clean_text(question)

        ids = [
            self.word2idx.get(t, 1)
            for t in tokens
        ]

        length = min(len(ids), self.max_len)

        ids = ids[:self.max_len]

        while len(ids) < self.max_len:
            ids.append(0)

        return torch.tensor(ids), length

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # =====================================================
        # IMAGE
        # =====================================================
        image_id = row["image_id"]

        img_name = (
            f"COCO_train2014_{str(image_id).zfill(12)}.jpg"
        )

        with zipfile.ZipFile(self.zip_path, 'r') as z:

            with z.open(f"train2014/{img_name}") as f:

                img = Image.open(f).convert("RGB")

        img = self.transform(img)

        # =====================================================
        # QUESTION
        # =====================================================
        question = row["question"]

        q_ids, length = self.tokenize_question(question)

        # =====================================================
        # ANSWER
        # =====================================================
        answer = row["answer"]

        label = self.ans2idx.get(
           answer,
          self.ans2idx["<UNK>"]
        )

        return (
            img,
            q_ids,
            torch.tensor(length),
            torch.tensor(label)
        )

In [12]:
train_dataset = VQADataset(
    train_df,
    word2idx,
    ans2idx,
    IMG_ZIP_PATH,
    MAX_LEN
)

val_dataset = VQADataset(
    val_df,
    word2idx,
    ans2idx,
    IMG_ZIP_PATH,
    MAX_LEN
)

test_dataset = VQADataset(
    test_df,
    word2idx,
    ans2idx,
    IMG_ZIP_PATH,
    MAX_LEN
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 10500
Val samples: 2250
Test samples: 2250


In [13]:
#TẠO DATALOADER
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("✅ DataLoader created")

✅ DataLoader created


In [12]:
#test thử DataLoader
images, questions, lengths, labels = next(iter(train_loader))

print(images.shape)
print(questions.shape)
print(lengths.shape)
print(labels.shape)

torch.Size([64, 3, 224, 224])
torch.Size([64, 20])
torch.Size([64])
torch.Size([64])


In [14]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# =========================================================
# VQA MODEL
# =========================================================
class VQAModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        num_answers,
        embed_dim=256,
        lstm_hidden=512,
        img_feat_dim=512,
        dropout=0.3
    ):
        super().__init__()

        # =====================================================
        # IMAGE ENCODER (ResNet50)
        # =====================================================
        resnet = resnet50(weights=ResNet50_Weights.DEFAULT)

        # Freeze CNN weights
        for param in resnet.parameters():
            param.requires_grad = False

        # Thay thế lớp FC cuối cùng bằng nn.Identity để lấy 2048 dims từ pooling
        self.cnn = resnet
        self.cnn.fc = nn.Identity()

        # Image projection (Thay BatchNorm1d bằng LayerNorm)
        self.img_fc = nn.Sequential(
            nn.Linear(2048, img_feat_dim),
            nn.LayerNorm(img_feat_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # =====================================================
        # QUESTION ENCODER
        # =====================================================
        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=lstm_hidden,
            batch_first=True
        )

        # =====================================================
        # CLASSIFIER
        # =====================================================
        fusion_dim = img_feat_dim + lstm_hidden

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_answers)
        )

    # =========================================================
    # FORWARD
    # =========================================================
    def forward(self, images, questions, lengths):

        # =====================================================
        # IMAGE FEATURES
        # =====================================================
        with torch.no_grad():
            # Output shape từ resnet lúc này trực tiếp là (B, 2048)
            img_feat = self.cnn(images)

        # (B, img_feat_dim)
        img_feat = self.img_fc(img_feat)

        # =====================================================
        # QUESTION FEATURES
        # =====================================================
        embedded = self.embedding(questions)

        # Cách an toàn để lấy hidden state cuối cùng chuẩn thứ tự batch:
        # Sử dụng output và lấy token tại vị trí thực tế của length (trừ đi 1)
        # Hoặc dùng pad_packed_sequence để khôi phục thứ tự gốc:
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, (hidden, _) = self.lstm(packed)

        # Khôi phục lại đúng thứ tự batch ban đầu ban đầu của câu hỏi
        output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)

        # Lấy hidden state tại vị trí kết thúc thực tế của từng câu
        batch_size = questions.size(0)
        q_feat = output[torch.arange(batch_size), lengths - 1]

        # =====================================================
        # FUSION & CLASSIFICATION
        # =====================================================
        fused = torch.cat([img_feat, q_feat], dim=1)
        output = self.classifier(fused)

        return output

In [1]:
# train mô hình
import os
import torch
import torch.nn as nn

from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# =========================================================
# DEVICE
# =========================================================
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# =========================================================
# SAVE PATH
# =========================================================
save_path = "/content/drive/MyDrive/VQA/Model/ResNet_LSTM_15k.pth"

os.makedirs(
    os.path.dirname(save_path),
    exist_ok=True
)

# =========================================================
# MODEL
# =========================================================
model = VQAModel(
    vocab_size=len(word2idx),
    num_answers=len(ans2idx)
).to(device)

# =========================================================
# LOSS FUNCTION
# =========================================================
criterion = nn.CrossEntropyLoss()

# =========================================================
# OPTIMIZER
# =========================================================
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4
)

# =========================================================
# LEARNING RATE SCHEDULER
# =========================================================
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# =========================================================
# TRAIN CONFIG
# =========================================================
EPOCHS = 20

best_val_loss = float("inf")

# Early stopping
patience = 7
counter = 0

# =========================================================
# HISTORY
# =========================================================
history = {

    "train_loss": [],

    "val_loss": [],

    "val_acc": [],

    "val_precision": [],

    "val_recall": [],

    "val_f1": []

}

# =========================================================
# TRAIN LOOP
# =========================================================
for epoch in range(EPOCHS):

    # =====================================================
    # TRAIN PHASE
    # =====================================================
    model.train()

    running_train_loss = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for images, questions, lengths, labels in train_bar:

        # -------------------------------------------------
        # TO DEVICE
        # -------------------------------------------------
        images = images.to(device)

        questions = questions.to(device)

        labels = labels.to(device)

        # -------------------------------------------------
        # FORWARD
        # -------------------------------------------------
        outputs = model(
            images,
            questions,
            lengths
        )

        loss = criterion(
            outputs,
            labels
        )

        # -------------------------------------------------
        # BACKWARD
        # -------------------------------------------------
        optimizer.zero_grad()

        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        # -------------------------------------------------
        # LOGGING
        # -------------------------------------------------
        running_train_loss += loss.item()

        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_train_loss = (
        running_train_loss / len(train_loader)
    )

    history["train_loss"].append(
        avg_train_loss
    )

    # =====================================================
    # VALIDATION PHASE
    # =====================================================
    model.eval()

    running_val_loss = 0.0

    all_labels = []

    all_preds = []

    with torch.inference_mode():

        for images, questions, lengths, labels in val_loader:

            # ---------------------------------------------
            # TO DEVICE
            # ---------------------------------------------
            images = images.to(device)

            questions = questions.to(device)

            labels = labels.to(device)

            # ---------------------------------------------
            # FORWARD
            # ---------------------------------------------
            outputs = model(
                images,
                questions,
                lengths
            )

            loss = criterion(
                outputs,
                labels
            )

            running_val_loss += loss.item()

            preds = outputs.argmax(dim=1)

            # ---------------------------------------------
            # STORE RESULTS
            # ---------------------------------------------
            all_labels.extend(
                labels.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

    avg_val_loss = (
        running_val_loss / len(val_loader)
    )

    # =====================================================
    # METRICS
    # =====================================================
    acc = accuracy_score(
        all_labels,
        all_preds
    )

    precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    # =====================================================
    # SAVE HISTORY
    # =====================================================
    history["val_loss"].append(
        avg_val_loss
    )

    history["val_acc"].append(
        acc
    )

    history["val_precision"].append(
        precision
    )

    history["val_recall"].append(
        recall
    )

    history["val_f1"].append(
        f1
    )

    # =====================================================
    # UPDATE LEARNING RATE
    # =====================================================
    scheduler.step(avg_val_loss)

    # =====================================================
    # CURRENT LEARNING RATE
    # =====================================================
    current_lr = optimizer.param_groups[0]["lr"]

    # =====================================================
    # SAVE BEST MODEL
    # =====================================================
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        counter = 0

        torch.save({

            "epoch":
                epoch + 1,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "train_loss":
                avg_train_loss,

            "val_loss":
                avg_val_loss,

            "accuracy":
                acc,

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1,

            "learning_rate":
                current_lr,

            "history":
                history,

            "word2idx":
                word2idx,

            "ans2idx":
                ans2idx

        }, save_path)

        print(
            f"\n✅ BEST MODEL SAVED "
            f"(Val Loss = {avg_val_loss:.4f} | "
            f"Acc = {acc:.4f})"
        )

    else:

        counter += 1

    # =====================================================
    # EARLY STOPPING
    # =====================================================
    if counter >= patience:

        print(
            f"\n🛑 EARLY STOPPING TRIGGERED "
            f"AT EPOCH {epoch+1}"
        )

        break

    # =====================================================
    # PRINT RESULTS
    # =====================================================
    print("\n==================================================")

    print(f"Epoch:         {epoch+1}/{EPOCHS}")

    print(f"Learning Rate: {current_lr:.6f}")

    print(f"Train Loss:    {avg_train_loss:.4f}")

    print(f"Val Loss:      {avg_val_loss:.4f}")

    print(f"Accuracy:      {acc:.4f}")

    print(f"Precision:     {precision:.4f}")

    print(f"Recall:        {recall:.4f}")

    print(f"F1 Score:      {f1:.4f}")

    print("==================================================\n")

print("✅ TRAINING FINISHED")

KeyboardInterrupt: 

In [9]:
import os
import torch
import torch.nn as nn

from tqdm import tqdm

from sklearn.metrics import accuracy_score

# =========================================================
# DEVICE
# =========================================================
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# =========================================================
# SAVE PATH
# =========================================================
save_path = "/content/drive/MyDrive/VQA/Model/ResNet_LSTM_15k.pth"

os.makedirs(
    os.path.dirname(save_path),
    exist_ok=True
)

# =========================================================
# MODEL
# =========================================================
model = VQAModel(
    vocab_size=len(word2idx),
    num_answers=len(ans2idx)
).to(device)

# =========================================================
# LOSS FUNCTION
# =========================================================
criterion = nn.CrossEntropyLoss()

# =========================================================
# OPTIMIZER
# =========================================================
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4
)

# =========================================================
# LR SCHEDULER
# =========================================================
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# =========================================================
# MIXED PRECISION
# =========================================================
scaler = torch.cuda.amp.GradScaler()

# =========================================================
# TRAIN CONFIG
# =========================================================
EPOCHS = 20

best_val_loss = float("inf")

# Early stopping
patience = 5
counter = 0

# =========================================================
# HISTORY
# =========================================================
history = {

    "train_loss": [],

    "val_loss": [],

    "val_acc": []

}

# =========================================================
# TRAIN LOOP
# =========================================================
for epoch in range(EPOCHS):

    # =====================================================
    # TRAIN PHASE
    # =====================================================
    model.train()

    running_train_loss = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for images, questions, lengths, labels in train_bar:

        # -------------------------------------------------
        # TO DEVICE
        # -------------------------------------------------
        images = images.to(device)

        questions = questions.to(device)

        labels = labels.to(device)

        # -------------------------------------------------
        # ZERO GRAD
        # -------------------------------------------------
        optimizer.zero_grad()

        # -------------------------------------------------
        # MIXED PRECISION FORWARD
        # -------------------------------------------------
        with torch.cuda.amp.autocast():

            outputs = model(
                images,
                questions,
                lengths
            )

            loss = criterion(
                outputs,
                labels
            )

        # -------------------------------------------------
        # BACKWARD
        # -------------------------------------------------
        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)

        scaler.update()

        # -------------------------------------------------
        # LOGGING
        # -------------------------------------------------
        running_train_loss += loss.item()

    avg_train_loss = (
        running_train_loss / len(train_loader)
    )

    history["train_loss"].append(
        avg_train_loss
    )

    # =====================================================
    # VALIDATION PHASE
    # =====================================================
    model.eval()

    running_val_loss = 0.0

    all_labels = []

    all_preds = []

    with torch.inference_mode():

        for images, questions, lengths, labels in val_loader:

            # ---------------------------------------------
            # TO DEVICE
            # ---------------------------------------------
            images = images.to(device)

            questions = questions.to(device)

            labels = labels.to(device)

            # ---------------------------------------------
            # FORWARD
            # ---------------------------------------------
            outputs = model(
                images,
                questions,
                lengths
            )

            loss = criterion(
                outputs,
                labels
            )

            running_val_loss += loss.item()

            preds = outputs.argmax(dim=1)

            # ---------------------------------------------
            # STORE RESULTS
            # ---------------------------------------------
            all_labels.extend(
                labels.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

    avg_val_loss = (
        running_val_loss / len(val_loader)
    )

    # =====================================================
    # ACCURACY ONLY
    # =====================================================
    acc = accuracy_score(
        all_labels,
        all_preds
    )

    # =====================================================
    # SAVE HISTORY
    # =====================================================
    history["val_loss"].append(
        avg_val_loss
    )

    history["val_acc"].append(
        acc
    )

    # =====================================================
    # UPDATE LR
    # =====================================================
    scheduler.step(avg_val_loss)

    # =====================================================
    # CURRENT LR
    # =====================================================
    current_lr = optimizer.param_groups[0]["lr"]

    # =====================================================
    # SAVE BEST MODEL
    # =====================================================
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        counter = 0

        torch.save({

            "epoch":
                epoch + 1,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "train_loss":
                avg_train_loss,

            "val_loss":
                avg_val_loss,

            "accuracy":
                acc,

            "learning_rate":
                current_lr,

            "history":
                history,

            "word2idx":
                word2idx,

            "ans2idx":
                ans2idx

        }, save_path)

        print(
            f"\n✅ BEST MODEL SAVED "
            f"(Val Loss = {avg_val_loss:.4f} | "
            f"Acc = {acc:.4f})"
        )

    else:

        counter += 1

    # =====================================================
    # EARLY STOPPING
    # =====================================================
    if counter >= patience:

        print(
            f"\n🛑 EARLY STOPPING "
            f"AT EPOCH {epoch+1}"
        )

        break

    # =====================================================
    # PRINT RESULT
    # =====================================================
    print("\n==================================================")

    print(f"Epoch:         {epoch+1}/{EPOCHS}")

    print(f"Learning Rate: {current_lr:.6f}")

    print(f"Train Loss:    {avg_train_loss:.4f}")

    print(f"Val Loss:      {avg_val_loss:.4f}")

    print(f"Accuracy:      {acc:.4f}")

    print("==================================================\n")

print("✅ TRAINING FINISHED")

Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 164MB/s]
/tmp/ipykernel_4078/1147017796.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
Epoch 1/20:   0%|          | 0/329 [00:00<?, ?it/s]/tmp/ipykernel_4078/1147017796.py:125: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_4078/980314037.py", line 100, in __getitem__
    label = self.ans2idx[answer]
            ~~~~~~~~~~~~^^^^^^^^
KeyError: 'california'
